# Preliminary setups


In [276]:
!pip install langchain_experimental
!pip install langchain-groq
!pip install ipykernel
!python -m ipykernel install --user --name=medical-agents --display-name "DANA system"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-groq]
Installed kernelspec medical-agents in /Users/natjs/Library/Jupyter/kernels/medical-agents


Here we define our api keys etc

In [279]:
import getpass
import os

def _set_if_undefined(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"Provide your {var}")

_set_if_undefined("OPENAI_API_KEY")
_set_if_undefined("TAVILY_API_KEY")
_set_if_undefined("GROQ_API_KEY")

Provide your GROQ_API_KEY ········


In [242]:
!pip install -U langgraph langchain-core langchain-community langchain-tavily langchain_openai langchain-anthropic langchain_google_genai

# All imports

The imports needed in this system to make sure we have our packages in place and it works 

In [291]:
from typing import Annotated, List, Dict, Optional, Literal, TypedDict
import os
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.graph import MessagesState, StateGraph, START, END
from langchain_core.language_models import BaseChatModel
from langgraph.types import Command
from langchain_tavily import TavilySearch
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langchain_experimental.utilities import PythonREPL
from langchain_groq import ChatGroq

# Define the tools


Define tools like web scrapping, writting a document, reading etc

In [302]:
@tool
def scrape_webpages(urls: List[str]) -> str:
    """User requests and bs4 to scrape the provided web pages for detailed information"""
    loader = WebBaseLoader(urls)
    docs = loader.load()
    return "\n\n".join(
        [
            f'<Document name="{doc.metadata.get("title", "")}">\n'
            f'{doc.page_content}\n'
            f'</Document>'
        ]
    )
@tool
def create_outline(
    points: Annotated[List[str], "List of main points or sections"],
    file_name: Annotated[str, "File path to save the outline"]
) -> Annotated[str, "Path of the saved outline file"]:
    """Create and save outline file"""
    os.makedirs(os.path.join(os.getcwd(), "temp"), exist_ok=True)
    file_to_use = os.path.join(os.getcwd(), "temp", file_name)
    with open(file_to_use, "w") as file:
        for i, point in enumerate(points):
            file.write(f"{i + 1}. {point}\n")
    return f"Outline saved to {file_name}"

@tool
def read_document(
    file_name: Annotated[str, "File path to read the document from"],
    start: Annotated[Optional[str], "The start line. Default is 0"] = None, 
    end: Annotated[Optional[str], "The end line. Defaul is None"] = None,
):
    """Read the specific document"""
    file_to_use = os.path.join(os.getcwd(), "temp", file_name)
    try:
        with open(file_to_use, "r") as file:
            lines = file.readlines()
            if start is None:
                start = 0
                
            start_idx = int(start) if start and str(start).isdigit() else 0
            end_idx = int(end) if end and str(end).isdigit() else None
            
            return "".join(lines[start_idx:end_idx])
    except FileNotFoundError:
        return f"Error: Document {file_name} not found"

@tool
def write_document(
    content: Annotated[str, "Text content to be written to the document"],
    file_name: Annotated[str, "File path to save the document"]
):
    """Create and save a text document"""
    os.makedirs(os.path.join(os.getcwd(), "temp"), exist_ok=True)
    file_to_use = os.path.join(os.getcwd(), "temp", file_name)
    with open(file_to_use, "w") as file:
        file.write(content)
    return f"Document saved to {file_name}"
                                                          

@tool
def edit_document(
    file_name: Annotated[str, "File path to save the document"],
    inserts: Annotated[Dict[str, str], "Dictionary where key is the line number and value is the text to be inserted at the line"]   
):
    """Edit a document by inserting text at specified line numbers"""
    file_to_use = os.path.join(os.getcwd(), "temp", file_name)
    try:
        with open(file_to_use, "r") as file:
            lines = file.readlines()
    except FileNotFoundError:
        return f"Error: Document {file_name} not found."

    try:
        parsed_inserts = {int(k): v for k, v in inserts.items()}
        sorted_inserts = sorted(parsed_inserts.items())
    except ValueError:
        return "Error: All keys in the inserts dictionary must be valid numbers."

    for line_number, text in sorted_inserts:
        if 1 <= line_number <= len(lines) + 1:
            lines.insert(line_number-1, text + "\n")
        else:
            return f"Error: line number {line_number} is out of range"
            
    with open(file_to_use, "w") as file:
        file.writelines(lines)

    return f"Document edited and saved to {file_name}"

repl = PythonREPL()

@tool
def python_repl_tool(code: Annotated[str, "The python code to execute to generate your chart"]):
    """Use this to execute python code. If you want to see the output of any value,
    you should print it with `print(...)`. This is visible to the user"""
    try:
        result = repl.run(code)
    except BaseException as e:
        return f"Failed to execute. Error: {repr(e)}"
    return f"Successfully executed: \n ```python \n{code}\n Stdout: {result}"
    

# Define the Supervisor

In [303]:
class State(MessagesState):
    next: str

def make_supervisor_node(llm: BaseChatModel, members: List[str]):
    options = ["FINISH"] + members
    system_prompt = (
        f" following workers: {members}. Given the following user request,"
        " respond with the worker to act next. Each worker will perform a"
        " task and respond with their results and status. When finished,"
        " respond with FINISH"
    )
    
    class Router(TypedDict):
        next: Literal[*options]
        
    def supervisor(state: State) -> Command[Literal[*members, "__end__"]]:
        messages = [
            {"role": "system", "content": system_prompt}
        ] + state["messages"]
    
        response = llm.with_structured_output(Router).invoke(messages)
        goto = response["next"]
        if goto == "FINISH":
            goto = END
        return Command(
            goto=goto, 
            update={"next": goto}
        )
    return supervisor
    

## Define the Agent Teams

### 1. Reader Agent - 

In [304]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)


tavily_tool = TavilySearch(max_results=3)

search_agent = create_react_agent(llm, tools=[tavily_tool])

def search_node(state: State) -> Command[Literal["supervisor"]]:
    result = search_agent.invoke(state)
    return Command(
        update={
            "messages": [
                HumanMessage(
                    content=result["messages"][-1].content, 
                    name="search"
                )
            ]
        },
        goto = "supervisor"
    )

web_scrapper_agent = create_react_agent(llm, tools=[scrape_webpages])
    
def web_scrapper_node(state: State) -> Command[Literal["supervisor"]]:
    result = web_scrapper_agent.invoke(state)
    return Command(
        update={
            "messages": [
                HumanMessage(
                    content=result["messages"][-1].content, 
                    name="web_scrapper"
                )
            ]
        },
        goto = "supervisor"
    )
research_supervisor_node = make_supervisor_node(llm, ["search", "web_scrapper"])

/var/folders/rp/bltyyw817g9593kj_7pxytnc0000gn/T/ipykernel_64511/255773831.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  search_agent = create_react_agent(llm, tools=[tavily_tool])
/var/folders/rp/bltyyw817g9593kj_7pxytnc0000gn/T/ipykernel_64511/255773831.py:25: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  web_scrapper_agent = create_react_agent(llm, tools=[scrape_webpages])


In [305]:
research_builder = StateGraph(State)
research_builder.add_node("supervisor", research_supervisor_node)
research_builder.add_node("search", search_node)
research_builder.add_node("web_scrapper", web_scrapper_node)

research_builder.add_edge(START, "supervisor")
research_graph = research_builder.compile()

### 2. Critic Agent - 

### 3. Teacher Agent - 

### 4. Obsidian Agent - 

### 5. Latex Agent - 

In [306]:
doc_writer_agent = create_react_agent(
    llm,
    tools = [write_document, edit_document, read_document],
    prompt="You can read, write and edit documents based on note taker's outlines. Don't ask follow up questions."
)

def doc_writer_node(state: State) -> Command[Literal["supervisor"]]:
    result = doc_writer_agent.invoke(state)
    return Command(
        update = {
            "messages": [
                HumanMessage(content = result["messages"][-1].content, name = "doc_writer")
            ]
        },
        goto = "supervisor",
    )
    
note_taking_agent = create_react_agent(
    llm,
    tools=[create_outline, read_document],
    prompt="You can read documents and create outlines for the document writer. Don't ask follow up questions."
)
        
def note_taking_node(state: State) -> Command[Literal["supervisor"]]:
    result = note_taking_agent.invoke(state)
    return Command(
        update = {
            "messages": [
                HumanMessage(
                    content = result["messages"][-1].content,
                    name = "note_taker"
                )
            ]
        },
        goto = "supervisor",
    )

chart_generating_agent = create_react_agent (
    llm,
    tools=[read_document, python_repl_tool],
)


def chart_generating_node(state: State) -> Command[Literal["supervisor"]]:
    result = chart_generating_agent.invoke(state)
    return Command(
        update = {
            "messages": [
                HumanMessage(
                    content = result["messages"][-1].content,
                    name = "chart_generator"
                )
            ]
        },
        goto = "supervisor",
    )

doc_writing_supervisor_node = make_supervisor_node(
    llm, ["doc_writer", "note_taker", "chart_generator"]
)

/var/folders/rp/bltyyw817g9593kj_7pxytnc0000gn/T/ipykernel_64511/404296440.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  doc_writer_agent = create_react_agent(
/var/folders/rp/bltyyw817g9593kj_7pxytnc0000gn/T/ipykernel_64511/404296440.py:18: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  note_taking_agent = create_react_agent(
/var/folders/rp/bltyyw817g9593kj_7pxytnc0000gn/T/ipykernel_64511/404296440.py:38: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  chart_generating_agent = create_re

In [307]:
writing_builder = StateGraph(State) # Reconstrucción grafo
writing_builder.add_node("supervisor", doc_writing_supervisor_node)
writing_builder.add_node("doc_writer", doc_writer_node)
writing_builder.add_node("note_taker", note_taking_agent)
writing_builder.add_node("chart_generator", chart_generating_node)

writing_builder.add_edge(START, "supervisor")
writing_graph = writing_builder.compile()

In [308]:
for s in writing_graph.stream(
    {
        "messages": [
            (
            "user",
            "Write an outline for a poem about dogs and after write the poem itself and store it"
            )
        ]
    },
        {"recursion_limit":30}
        ):
        print(s)
        print("---")
    
        

{'supervisor': {'next': 'doc_writer'}}
---
{'doc_writer': {'messages': [HumanMessage(content='The outline and poem have been written and saved to their respective files.', additional_kwargs={}, response_metadata={}, name='doc_writer', id='b385ab28-d7c5-4bb1-a81e-c1dc2f0657b3')]}}
---
{'supervisor': {'next': 'note_taker'}}
---
{'note_taker': {'messages': [HumanMessage(content='Write an outline for a poem about dogs and after write the poem itself and store it', additional_kwargs={}, response_metadata={}, id='d9fadb3c-1b36-4a2f-ace7-f19bbe82420c'), HumanMessage(content='The outline and poem have been written and saved to their respective files.', additional_kwargs={}, response_metadata={}, name='doc_writer', id='b385ab28-d7c5-4bb1-a81e-c1dc2f0657b3'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6ty4e6khx', 'function': {'arguments': '{"file_name":"poem_outline.txt","points":["Introduction","Body","Conclusion"]}', 'name': 'create_outline'}, 'type': 'function'}, {'id': '

# End-to-end Graph

## Implement the graph

## Visualize the graph

## Run the graph